# Single STAC Item: `open_item` and `open_cog`

Two single-item helpers in `lazycogs`, both lazy and read at the asset's native grid:

- `lazycogs.open_item` — stack several same-grid bands of one STAC item into a `(band, y, x)` DataArray.
- `lazycogs.open_cog` — read one Cloud-Optimized GeoTIFF at its native CRS, resolution, and shape.

First, find one low-cloud Sentinel-2 scene and configure a store for the public bucket (no credentials needed).

In [3]:
import rustac
from obstore.store import S3Store

import lazycogs

items = await rustac.search(
    href="https://earth-search.aws.element84.com/v1",
    collections=["sentinel-2-c1-l2a"],
    bbox=[4.8, 52.3, 5.0, 52.5],  # Amsterdam
    datetime="2023-06-01/2023-06-30",
    query={"eo:cloud_cover": {"lt": 10}},
    limit=1,
)
item = items[0]

store = S3Store(
    bucket="e84-earth-search-sentinel-data",
    region="us-west-2",
    skip_signature=True,
    virtual_hosted_style_request=True,
)

## Open a Multi-Band Item with `open_item`

`lazycogs.open_item` stacks several single-band assets of one STAC item into a `(band, y, x)` DataArray at their shared native grid. Here we request the `red` and `green` bands. Notice that both of these assets have the same grid. If we were to pick bands with different grids, `lazycogs.open_item` would raise an error, `lazycogs.open` would then be the correct function to use.

In [4]:
da = lazycogs.open_item(item, bands=["red", "green"], store=store)
da

<xarray.DataArray (band: 2, y: 10980, x: 10980)> Size: 482MB
array([[[1216, 1236, 1198, ..., 1468, 1463, 1450],
        [1211, 1231, 1210, ..., 1452, 1464, 1460],
        [1197, 1218, 1226, ..., 1456, 1458, 1467],
        ...,
        [1205, 1193, 1209, ..., 1776, 2122, 2232],
        [1185, 1187, 1254, ..., 1570, 1964, 2134],
        [1196, 1198, 1228, ..., 1564, 1697, 1570]],

       [[1301, 1338, 1316, ..., 1660, 1661, 1647],
        [1328, 1327, 1313, ..., 1662, 1652, 1662],
        [1314, 1327, 1327, ..., 1665, 1664, 1668],
        ...,
        [1323, 1312, 1328, ..., 1742, 1926, 1947],
        [1328, 1335, 1392, ..., 1658, 1850, 1904],
        [1325, 1366, 1372, ..., 1654, 1665, 1562]]],
      shape=(2, 10980, 10980), dtype=uint16)
Coordinates:
  * band         (band) <U5 40B 'red' 'green'
  * y            (y) float64 88kB 5.8e+06 5.8e+06 5.8e+06 ... 5.69e+06 5.69e+06
  * x            (x) float64 88kB 6e+05 6e+05 6e+05 ... 7.098e+05 7.098e+05
    spatial_ref  int64 8B 0
Indexes:
  ┌ x        RasterIndex (crs=EPSG:32631)
  └ y
Attributes:
    grid_mapping:  spatial_ref
    _FillValue:    0.0
    scale_factor:  0.0001
    add_offset:    -0.1

## Open a Single COG with `open_cog`

`lazycogs.open_cog` reads one Cloud-Optimized GeoTIFF in place — native CRS, resolution, and shape, with no reprojection. Here we open just the `nir` asset of the same item directly from its href.

In [5]:
nir = lazycogs.open_cog(item["assets"]["nir"]["href"], store=store)
nir

<xarray.DataArray (band: 1, y: 10980, x: 10980)> Size: 241MB
array([[[1164, 1182, 1181, ..., 4315, 4286, 4254],
        [1174, 1176, 1191, ..., 4307, 4340, 4299],
        [1166, 1177, 1194, ..., 4268, 4313, 4280],
        ...,
        [4155, 3839, 3721, ..., 3898, 3349, 3091],
        [4223, 4033, 3884, ..., 4019, 3483, 2989],
        [4137, 4081, 3784, ..., 4055, 3601, 3236]]],
      shape=(1, 10980, 10980), dtype=uint16)
Coordinates:
  * band         (band) int64 8B 1
  * y            (y) float64 88kB 5.8e+06 5.8e+06 5.8e+06 ... 5.69e+06 5.69e+06
  * x            (x) float64 88kB 6e+05 6e+05 6e+05 ... 7.098e+05 7.098e+05
    spatial_ref  int64 8B 0
Indexes:
  ┌ x        RasterIndex (crs=EPSG:32631)
  └ y
Attributes:
    grid_mapping:  spatial_ref
    _FillValue:    0.0
    scale_factor:  0.0001
    add_offset:    -0.1